# Anomaly detection with PyOD

Notebook version of [AnomalyDetectionUsingPyOD](https://github.com/JaminJeong/AnomalyDetectionUsingPyOD) (daily minimum temperatures, sliding windows, multiple detectors). Slots that originally required **Numba** or **torch** are filled with **sklearn substitute** detectors (clearly labeled—not the same algorithms as upstream PyOD). Temperatures are loaded with **`%%cribl_search`** using Cribl Search [`externaldata`](https://docs.cribl.io/search/externaldata/) (same public CSV as the upstream example—see [daily-min-temperatures.csv](https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv) and [Machine Learning Mastery](https://machinelearningmastery.com/time-series-datasets-for-machine-learning/)). Charts use **Plotly** (interactive pan/zoom, anomaly markers with hovers and index labels when sparse). [PyOD](https://github.com/yzhao062/pyod) is installed in **Setup** via `micropip`.

**Background (opens in new tab):** <a href="https://arxiv.org/abs/1910.02575" target="_blank" rel="noopener noreferrer">PyOD overview paper (arXiv)</a> · <a href="https://pyod.readthedocs.io/en/latest/" target="_blank" rel="noopener noreferrer">PyOD documentation</a> · <a href="https://scikit-learn.org/stable/modules/outlier_detection.html" target="_blank" rel="noopener noreferrer">scikit-learn outlier & novelty detection</a>.

## How to run

1. Run **Setup** once (downloads PyOD stack and Plotly; first run can take several minutes).
2. Run **Shared preprocessing** (Python helpers and constants).
3. Run **Load temperatures** (`%%cribl_search` with `externaldata`) — requires this notebook to run **inside a hosted Cribl app** where Search is available (like other `%%cribl_search` examples). For a VPC flow IP watchlist hunt (lookup + join + timestats), see **`Threat_Hunting_Playbook.ipynb`**.
4. Run **Materialize dataframe**, then **train/test windows**, then each **detector** cell (each shows its own interactive Plotly chart). Sections titled **sklearn substitute for …** replace PyOD models that need **Numba** or **PyTorch** with scikit-learn (or pure NumPy) alternatives—they are **not** the same algorithms as upstream PyOD. **Variational Auto Encoder** and **Auto Encoder** slots use PCA / MLP reconstruction scores instead of torch. The upstream **LOCI** screenshots are not part of the upstream script's active models; this notebook keeps the same **18** slots as `anomaly_detection_all_model.py`.

Under **Detectors**, each method section includes **Learn more** links that open **official docs and papers** in a **new browser tab** (the notebook app allows `target=_blank` on trusted markdown links).

### Setup

Installs **SciPy**, **scikit-learn**, **joblib**, and **Plotly** (`5.24.1`, pinned for Pyodide—same as the Visualisations example), then **PyOD** with `deps=False` (PyPI also declares **Numba**, which has no Pyodide wheel—PyOD models that import Numba are not used here; see **sklearn substitute** sections instead). Requires PyPI/jsDelivr (allow-listed in `proxies.yml`).

In [ ]:
import micropip

# PyPI declares `numba` for pyod, but Numba has no Pyodide wheel — install the
# scientific stack explicitly, then pyod without transitive deps.
# Pin versions to the Pyodide full-distribution wheel set for this app
# (see `PYODIDE_RELEASE` / bundled `pyodide-lock.json`; revisit pins when bumping Pyodide).
await micropip.install([
    'scipy==1.14.1',
    'scikit-learn==1.7.0',
    'joblib==1.4.2',
])
# Plotly 6+ pulls narwhals paths that are awkward with pandas in Pyodide — pin like Visualisations.ipynb.
await micropip.install(['plotly==5.24.1'])
await micropip.install(['pyod==2.0.5'], deps=False)

import warnings

# Interactive charts use MIME output only (`fig.show()`), not static export — silence Plotly vs Kaleido wheel mismatch.
warnings.filterwarnings(
    'ignore',
    category=UserWarning,
    message=r'.*not compatible with this version of Kaleido.*',
)
# sklearn/threadpoolctl probes native libs via Pyodide — JsProxy API deprecation noise.
warnings.filterwarnings(
    'ignore',
    category=RuntimeWarning,
    message=r'.*JsProxy\.as_object_map\(\) is deprecated.*',
)


### Shared preprocessing

Imports, constants (`window_size`, `contamination`), helper functions for sliding windows and **Plotly** charts (`plot_anomalies` — anomaly markers show **model score** plus a short **why** line from `decision_function` / detector-specific margins), and **sklearn adapter classes** for substitute detectors. Also builds the **neighbor grid** used by the **sklearn substitute for LSCP** cell. Run **Setup** above first so `pyod` and `plotly` are available.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

window_size = 30
contamination = 0.25
outliers_fraction = contamination
random_state = 42


def _score_hint_for_detector(clf):
    """One-line explanation of what the hover score measures."""
    name = clf.__class__.__name__
    if name == 'SklearnNegLabelsToPyOD':
        est = clf._e
        if hasattr(est, 'named_steps'):
            _, last = est.steps[-1]
            ln = last.__class__.__name__
            if ln == 'LocalOutlierFactor':
                return 'Novelty LOF (more negative => farther from training neighbors)'
            return f'{ln} stage — see sklearn.decision_function'
        ln = est.__class__.__name__
        return {
            'OneClassSVM': 'RBF boundary side (neg => outside normal region)',
            'SGDOneClassSVM': 'Linear OC-SVM margin',
            'IsolationForest': 'Isolation score (lower => more anomalous in forest)',
        }.get(ln, 'Sklearn decision_function')
    hints = {
        'GMMLogDensityOutlier': 'Density gap: cutoff − log p under fitted mixture',
        'PCAMSEReconstructionOutlier': 'PCA recon MSE above train threshold',
        'MLPIdentityReconstructionOutlier': 'MLP identity recon error above threshold',
        'LOFMajorityVoteOutlier': 'LOF models voting outlier past majority',
        'IForest': 'PyOD isolation forest score',
        'KNN': 'PyOD k-neighbor distance score',
        'LOF': 'PyOD local outlier factor score',
        'MCD': 'PyOD minimum covariance determinant distance',
        'OCSVM': 'PyOD one-class SVM score',
        'PCA': 'PyOD PCA-based outlier score',
        'COF': 'PyOD connectivity-based outlier factor',
    }
    return hints.get(name, 'Outlier score (detector.decision_function)')


def _detector_scores(clf, X):
    if not hasattr(clf, 'decision_function'):
        return None, None
    try:
        s = np.asarray(clf.decision_function(X), dtype=float).reshape(-1)
        return s, _score_hint_for_detector(clf)
    except Exception:
        return None, None


def make_data_sampling(data, window_size):
    n_samples = len(data)
    if n_samples < window_size:
        raise ValueError('window size is too long for series')
    return np.array([data[i : i + window_size] for i in range(n_samples - window_size + 1)])


def plot_anomalies(title, real_value, y_pred, scores=None, score_hint=None):
    import plotly.graph_objects as go

    x = np.arange(len(real_value), dtype=float)
    rv = np.asarray(real_value, dtype=float)
    pred = np.asarray(y_pred)
    mask = pred == 1
    ax_idx = x[mask]
    ay = rv[mask]
    sc = np.asarray(scores, dtype=float).reshape(-1) if scores is not None else None

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x,
            y=rv,
            mode='lines',
            name='real',
            hovertemplate='index=%{x:.0f}<br>temp=%{y:.2f}<extra></extra>',
        )
    )

    n_anom = int(np.sum(mask))
    label_cap = 36
    if n_anom > 0:
        mode = 'markers+text' if n_anom <= label_cap else 'markers'
        if sc is not None and sc.shape[0] == len(rv):
            sc_a = sc[mask]
            customdata = np.column_stack([ax_idx, ay, sc_a])
            hover = (
                '<b>anomaly</b><br>idx=%{customdata[0]:.0f}<br>temp=%{customdata[1]:.2f}<br>'
                '<b>score</b>=%{customdata[2]:.4g}<br><i>'
                + (score_hint or '')
                + '</i><extra></extra>'
            )
            text = (
                [f'{int(i)}<br><span style="font-size:9px">s={v:.3g}</span>' for i, v in zip(ax_idx, sc_a)]
                if n_anom <= label_cap
                else None
            )
            fig.add_trace(
                go.Scatter(
                    x=ax_idx,
                    y=ay,
                    mode=mode,
                    name='anomaly',
                    marker=dict(size=11, color='rgba(140,140,140,0.9)', line=dict(width=2, color='red')),
                    text=text,
                    textposition='top center',
                    customdata=customdata,
                    hovertemplate=hover,
                )
            )
        else:
            text = [str(int(i)) for i in ax_idx] if n_anom <= label_cap else None
            fig.add_trace(
                go.Scatter(
                    x=ax_idx,
                    y=ay,
                    mode=mode,
                    name='anomaly',
                    marker=dict(size=11, color='rgba(140,140,140,0.9)', line=dict(width=2, color='red')),
                    text=text,
                    textposition='top center',
                    hovertemplate='anomaly<br>index=%{x:.0f}<br>temp=%{y:.2f}<extra></extra>',
                )
            )

    title_layout = dict(text=title, x=0.01, xanchor='left')
    if sc is not None and score_hint:
        title_layout['subtitle'] = dict(text=score_hint, font=dict(size=12))
    fig.update_layout(
        title=title_layout,
        height=400,
        margin=dict(l=48, r=24, t=64, b=44),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        xaxis_title='window index (test)',
        yaxis_title='temp (last step)',
        hovermode='closest',
    )
    fig.show()


def run_detector(title, clf):
    clf.fit(df_data_train)
    y_pred = clf.predict(df_data_test)
    real_value = df_data_test[:, -1]
    scores, hint = _detector_scores(clf, df_data_test)
    plot_anomalies(title, real_value, y_pred, scores=scores, score_hint=hint)


class SklearnNegLabelsToPyOD:
    # Wrap sklearn: predict 1=inlier, -1=outlier → PyOD-style 1=anomaly, 0=normal

    def __init__(self, estimator):
        self._e = estimator

    def fit(self, X):
        self._e.fit(X)
        return self

    def predict(self, X):
        y = self._e.predict(X)
        return np.where(y == -1, 1, 0)

    def decision_function(self, X):
        return self._e.decision_function(X)


class GMMLogDensityOutlier:
    # Low Gaussian log-density (cluster-flavored CBLOF substitute)

    def __init__(self, contamination, random_state, n_components=15):
        self.contamination = float(contamination)
        self.random_state = random_state
        self.n_components = int(n_components)
        self._gmm = None
        self._threshold = None

    def fit(self, X):
        from sklearn.mixture import GaussianMixture

        n_max = max(1, min(self.n_components, X.shape[0] // 3, X.shape[0] - 1))
        self._gmm = GaussianMixture(
            n_components=n_max,
            covariance_type='full',
            random_state=self.random_state,
            reg_covar=1e-6,
            max_iter=200,
        )
        self._gmm.fit(X)
        logp = self._gmm.score_samples(X)
        self._threshold = np.quantile(logp, self.contamination)
        return self

    def predict(self, X):
        logp = self._gmm.score_samples(X)
        return (logp < self._threshold).astype(int)

    def decision_function(self, X):
        logp = self._gmm.score_samples(X)
        return self._threshold - logp


class PCAMSEReconstructionOutlier:
    # High PCA reconstruction MSE (linear VAE-style substitute)

    def __init__(self, contamination, n_components=8, random_state=42):
        self.contamination = float(contamination)
        self.n_components = int(n_components)
        self.random_state = random_state
        self._pca = None
        self._thr = None

    def fit(self, X):
        from sklearn.decomposition import PCA

        ncomp = max(1, min(self.n_components, X.shape[1] - 1, max(1, X.shape[0] - 2)))
        self._pca = PCA(n_components=ncomp, random_state=self.random_state)
        self._pca.fit(X)
        z = self._pca.transform(X)
        recon = self._pca.inverse_transform(z)
        err = np.mean((X - recon) ** 2, axis=1)
        self._thr = np.quantile(err, 1.0 - self.contamination)
        return self

    def predict(self, X):
        z = self._pca.transform(X)
        recon = self._pca.inverse_transform(z)
        err = np.mean((X - recon) ** 2, axis=1)
        return (err > self._thr).astype(int)

    def decision_function(self, X):
        z = self._pca.transform(X)
        recon = self._pca.inverse_transform(z)
        err = np.mean((X - recon) ** 2, axis=1)
        return err - self._thr


class MLPIdentityReconstructionOutlier:
    # Small MLP identity reconstruction (autoencoder substitute)

    def __init__(self, contamination, max_iter=400, random_state=42):
        self.contamination = float(contamination)
        self.max_iter = int(max_iter)
        self.random_state = random_state
        self._mlp = None
        self._thr = None

    def fit(self, X):
        from sklearn.neural_network import MLPRegressor

        n_in = X.shape[1]
        h = max(2, min(32, n_in * 2))
        self._mlp = MLPRegressor(
            hidden_layer_sizes=(h, h),
            activation='relu',
            max_iter=self.max_iter,
            tol=1e-3,
            random_state=self.random_state,
            early_stopping=True,
            validation_fraction=0.12,
            n_iter_no_change=15,
        )
        self._mlp.fit(X, X)
        recon = self._mlp.predict(X)
        err = np.mean((X - recon) ** 2, axis=1)
        self._thr = np.quantile(err, 1.0 - self.contamination)
        return self

    def predict(self, X):
        recon = self._mlp.predict(X)
        err = np.mean((X - recon) ** 2, axis=1)
        return (err > self._thr).astype(int)

    def decision_function(self, X):
        recon = self._mlp.predict(X)
        err = np.mean((X - recon) ** 2, axis=1)
        return err - self._thr


class LOFMajorityVoteOutlier:
    # Majority vote over novelty LOF models (LSCP-style substitute)

    def __init__(self, neighbor_values, contamination):
        from sklearn.neighbors import LocalOutlierFactor

        self._estimators = []
        for n in neighbor_values:
            nn = int(max(2, int(n)))
            self._estimators.append(
                LocalOutlierFactor(n_neighbors=nn, novelty=True, contamination=contamination)
            )

    def fit(self, X):
        for e in self._estimators:
            e.fit(X)
        return self

    def predict(self, X):
        preds = np.stack([e.predict(X) for e in self._estimators], axis=0)
        votes_out = np.sum(preds == -1, axis=0)
        need = preds.shape[0] // 2 + 1
        return (votes_out >= need).astype(int)

    def decision_function(self, X):
        preds = np.stack([e.predict(X) for e in self._estimators], axis=0)
        votes_out = np.sum(preds == -1, axis=0).astype(float)
        need = preds.shape[0] // 2 + 1
        return votes_out - (need - 1)


### Load temperatures (`externaldata`)

Fetches the CSV through **[`externaldata`](https://docs.cribl.io/search/externaldata/)** so retrieval happens in **Cribl Search**, not via `pd.read_csv` in Pyodide. **`preview=true`** matches structured columns (typically **`Temp`** and **`Date`**) on the wire; **Materialize** reads **`Temp`** directly and sorts by **`Date`** when present. If you still get a single raw column (`Event` / `_raw`), the notebook falls back to CSV line parsing. Uses datatype **`CSV Datatypes`** — if Search rejects it on your tenant, try **`CSV`** (v1 stock) or your admin's comma-separated datatype.

In [ ]:
%%cribl_search var=temp_df lang=kql limit=0 preview=true earliest=-1h latest=now
externaldata
[
  "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
]
with(
  datatype="CSV Datatypes"
)

In [ ]:
import io

import pandas as pd

# With preview=true and typed CSV, Search usually returns structured columns (`Temp` or `Temperature`, often `Date`).
_cl = {str(c).strip().lower(): c for c in temp_df.columns}
_tcol = next((_cl[k] for k in ('temp', 'temperature') if k in _cl), None)

if _tcol is not None:
    tcol = _tcol
    df_data = pd.DataFrame({'Temp': temp_df[tcol]})
    _dcol = next((_cl[k] for k in ('date', 'time', '_time') if k in _cl), None)
    if _dcol is not None:
        order = pd.to_datetime(temp_df[_dcol], errors='coerce')
        df_data = df_data.assign(_sort=order).sort_values('_sort').drop(columns=['_sort'])
    df_data['Temp'] = pd.to_numeric(df_data['Temp'], errors='coerce')
    df_data = df_data.dropna(subset=['Temp']).reset_index(drop=True)
else:
    # Legacy: whole CSV line in one field — rebuild and parse.
    raw_col = _cl.get('event') or _cl.get('_raw')
    if raw_col is None:
        raw_col = next((c for c in temp_df.columns if str(c).startswith('_')), None)
    if raw_col is None:
        skip_t = {'time', '_time'}
        rest = [c for c in temp_df.columns if str(c).lower() not in skip_t]
        raw_col = rest[-1] if rest else temp_df.columns[-1]
    lines = temp_df[raw_col].astype(str).str.strip()
    lines = lines[lines.str.len() > 0]
    hdr = '"Date","Temp"'
    lines = lines[lines != hdr]
    df_data = pd.read_csv(io.StringIO(hdr + '\n' + '\n'.join(lines.tolist())))
    df_data.columns = [str(c).strip() for c in df_data.columns]
    if 'Temp' not in df_data.columns:
        raise ValueError(
            'Expected a Temp column from Search or CSV rows in Event/_raw; got '
            + repr(list(temp_df.columns))
        )
    df_data['Temp'] = pd.to_numeric(df_data['Temp'], errors='coerce')
    df_data = df_data.dropna(subset=['Temp']).reset_index(drop=True)

df_data.head()

In [ ]:
raw_train, raw_test = train_test_split(
    np.array(df_data['Temp']), test_size=0.2, shuffle=False
)

n_train_windows = len(raw_train) - window_size + 1
if n_train_windows < 2:
    raise ValueError(f'Not enough training points for window_size={window_size}; got {len(raw_train)} temps')

# sklearn / pyod require n_neighbors < n_samples (training rows are sliding windows)
_nn_cap = max(1, n_train_windows - 1)


def _cap_nn(k: int) -> int:
    return max(1, min(int(k), _nn_cap))


LOF_N = _cap_nn(35)
COF_N = LOF_N

_nn_grid = (5, 10, 15, 20, 25, 30, 35, 40, 45, 50)
lscp_neighbor_values = []
_seen = set()
for k in _nn_grid:
    v = _cap_nn(k)
    if v not in _seen:
        _seen.add(v)
        lscp_neighbor_values.append(v)
if len(lscp_neighbor_values) < 2:
    lscp_neighbor_values = [_nn_cap, max(1, _nn_cap // 2)]

df_data_train = make_data_sampling(raw_train, window_size)
df_data_test = make_data_sampling(raw_test, window_size)
print(
    'train windows',
    df_data_train.shape,
    'test windows',
    df_data_test.shape,
    'LOF_N',
    LOF_N,
    'LSCP sklearn neighbor grid',
    lscp_neighbor_values,
)


### Train series (subsampled)

Same idea as `GenGraph('train')` in the upstream `save_graph.py`.

In [ ]:
import plotly.graph_objects as go

_sub = raw_train[::30]
_x = np.arange(len(_sub))
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=_x,
        y=_sub,
        mode='lines',
        name='train',
        hovertemplate='subsample idx=%{x:.0f}<br>temp=%{y:.2f}<extra></extra>',
    )
)
fig.update_layout(title='Train temperatures (every 30th point)', height=320, margin=dict(t=48, b=40))
fig.show()

### Test series (subsampled)

Same idea as `GenGraph('test')`.

In [ ]:
import plotly.graph_objects as go

_sub = raw_test[::30]
_x = np.arange(len(_sub))
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=_x,
        y=_sub,
        mode='lines',
        name='test',
        hovertemplate='subsample idx=%{x:.0f}<br>temp=%{y:.2f}<extra></extra>',
    )
)
fig.update_layout(title='Test temperatures (every 30th point)', height=320, margin=dict(t=48, b=40))
fig.show()

## Detectors

Each section fits **one** model on `df_data_train`, scores `df_data_test`, and shows a **Plotly** chart of the **last timestep** of each window with anomaly markers and hovers (matching `BasicGenGraph` in the upstream project). Headings **sklearn substitute for …** use scikit-learn (or a small sklearn MLP) instead of the original PyOD model when that model would require **Numba** or **PyTorch** in the browser. Under each heading, a short paragraph explains what the upstream method does and what this notebook runs instead (or which PyOD class is used).

**Topic hubs (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.html" target="_blank" rel="noopener noreferrer">PyOD model API</a> · <a href="https://scikit-learn.org/stable/modules/outlier_detection.html" target="_blank" rel="noopener noreferrer">sklearn user guide</a>.

### sklearn substitute for Angle-based Outlier Detector (ABOD)

**Original idea (PyOD):** ABOD scores points using angular relationships between vectors in feature space, so outliers are directions that disagree with most of the data. **This cell:** we use a **nonlinear One-Class SVM with an RBF kernel**, which learns a smooth boundary around the bulk of training windows instead of angular scores. It is a standard, WASM-friendly density-envelope method—not the same math as ABOD, but it targets the same goal: windows whose pattern of temperatures is unlike the majority.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.abod.html" target="_blank" rel="noopener noreferrer">PyOD ABOD</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.svm.OneClassSVM.html" target="_blank" rel="noopener noreferrer">sklearn OneClassSVM</a>

In [ ]:
title = 'sklearn substitute for Angle-based Outlier Detector (ABOD) (RBF One-Class SVM)'
from sklearn.svm import OneClassSVM

clf = SklearnNegLabelsToPyOD(OneClassSVM(kernel='rbf', gamma='scale', nu=outliers_fraction))
run_detector(title, clf)


### sklearn substitute for Cluster-based Local Outlier Factor (CBLOF)

**Original idea (PyOD):** CBLOF clusters the data, then compares each point to its cluster context so dense clusters and sparse clusters are treated fairly for local outliers. **This cell:** we fit a **Gaussian mixture** on training windows and treat **low mixture log-density** (under the learned components) as anomalous—cluster structure plus a density score, without PyOD’s cluster-wise LOF machinery or Numba.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.cblof.html" target="_blank" rel="noopener noreferrer">PyOD CBLOF</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html" target="_blank" rel="noopener noreferrer">sklearn GaussianMixture</a>

In [ ]:
title = 'sklearn substitute for Cluster-based Local Outlier Factor (CBLOF) (Gaussian mixture log-density)'
clf = GMMLogDensityOutlier(outliers_fraction, random_state, n_components=15)
run_detector(title, clf)


### sklearn substitute for Feature Bagging

**Original idea (PyOD):** Feature Bagging trains base detectors on random subsets of features and combines their opinions so no single feature dimension dominates. **This cell:** we run **PCA** to a fixed-size linear subspace, then **novelty Local Outlier Factor** on the reduced vectors—reduction plus local density in one pipeline. It is an interpretable stand-in for “subspace + ensemble of local views,” not a literal feature-bagging implementation.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.feature_bagging.html" target="_blank" rel="noopener noreferrer">PyOD Feature Bagging</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html" target="_blank" rel="noopener noreferrer">sklearn PCA</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.LocalOutlierFactor.html" target="_blank" rel="noopener noreferrer">sklearn LocalOutlierFactor</a>

In [ ]:
title = 'sklearn substitute for Feature Bagging (PCA + novelty LOF on reduced space)'
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline

n_pca = max(2, min(10, df_data_train.shape[1] - 1))
nn_fb = max(2, min(LOF_N, df_data_train.shape[0] - 1))
pipe = Pipeline(
    [
        ('pca', PCA(n_components=n_pca, random_state=random_state)),
        (
            'lof',
            LocalOutlierFactor(n_neighbors=nn_fb, novelty=True, contamination=outliers_fraction),
        ),
    ]
)
clf = SklearnNegLabelsToPyOD(pipe)
run_detector(title, clf)


### sklearn substitute for Histogram-base Outlier Detection (HBOS)

**Original idea (PyOD):** HBOS builds histograms along features and flags points that land in rare bins—fast, axis-aligned density sketches. **This cell:** we use **scikit-learn’s Isolation Forest**, which isolates points using random axis-aligned splits; trees indirectly approximate multivariate density, so the story is “fast tree-based rarity” rather than explicit histogram bins, but it runs entirely on wheels Pyodide already supports.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.hbos.html" target="_blank" rel="noopener noreferrer">PyOD HBOS</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html" target="_blank" rel="noopener noreferrer">sklearn IsolationForest</a>

In [ ]:
title = 'sklearn substitute for Histogram-base Outlier Detection (HBOS) (IsolationForest)'
from sklearn.ensemble import IsolationForest

clf = SklearnNegLabelsToPyOD(
    IsolationForest(contamination=outliers_fraction, random_state=random_state, n_estimators=200)
)
run_detector(title, clf)


### Isolation Forest

**Algorithm:** Isolation Forest randomly partitions feature space with trees; points that require few splits to isolate are treated as anomalies because they sit in sparse regions of the data. **This cell:** uses **PyOD’s `IForest`** on sliding windows of temperatures, matching the upstream notebook’s tree-based baseline while staying compatible with the pinned scientific stack in Pyodide.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.iforest.html" target="_blank" rel="noopener noreferrer">PyOD IForest</a> · <a href="https://scikit-learn.org/stable/modules/outlier_detection.html#isolation-forest" target="_blank" rel="noopener noreferrer">sklearn: Isolation Forest</a>

In [ ]:
title = 'Isolation Forest'
try:
    from pyod.models.iforest import IForest
    clf = IForest(contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### K Nearest Neighbors (KNN)

**Algorithm:** Classic distance-based kNN outlier detection compares each point to its *k*-nearest neighbors in feature space; isolated or distant windows get high scores. **This cell:** **PyOD `KNN`** with default aggregation—good when local neighborhoods of the multivariate window vectors carry the signal for cold snaps or unusual short patterns.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.knn.html" target="_blank" rel="noopener noreferrer">PyOD KNN</a>

In [ ]:
title = 'K Nearest Neighbors (KNN)'
try:
    from pyod.models.knn import KNN
    clf = KNN(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Average KNN

**Algorithm:** Instead of a single neighbor distance, average KNN smooths the score by averaging distances to the *k* neighbors, reducing noise from one unlucky neighbor. **This cell:** **PyOD `KNN` with `method='mean'`**—slightly more stable local density view for the same window features.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.knn.html" target="_blank" rel="noopener noreferrer">PyOD KNN (mean aggregation)</a>

In [ ]:
title = 'Average KNN'
try:
    from pyod.models.knn import KNN
    clf = KNN(method='mean', contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Median KNN

**Algorithm:** Median KNN uses the median of neighbor distances, which is robust if a few neighbors are themselves unusual but the bulk of the neighborhood is still representative. **This cell:** **PyOD `KNN` with `method='median'`**—a robust local-distance variant on the temperature windows.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.knn.html" target="_blank" rel="noopener noreferrer">PyOD KNN (median aggregation)</a>

In [ ]:
title = 'Median KNN'
try:
    from pyod.models.knn import KNN
    clf = KNN(method='median', contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Local Outlier Factor (LOF)

**Algorithm:** LOF compares each point’s local reachability density to that of its neighbors; points in sparser pockets than their peers are outliers. **This cell:** **PyOD `LOF`** with **`LOF_N`** neighbors capped from the training window count so `n_neighbors` stays valid—local contrast on the same sliding-window representation as the rest of the notebook.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.lof.html" target="_blank" rel="noopener noreferrer">PyOD LOF</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.LocalOutlierFactor.html" target="_blank" rel="noopener noreferrer">sklearn LocalOutlierFactor</a>

In [ ]:
title = 'Local Outlier Factor (LOF)'
try:
    from pyod.models.lof import LOF
    clf = LOF(n_neighbors=LOF_N, contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Minimum Covariance Determinant (MCD)

**Algorithm:** MCD estimates a robust covariance ellipsoid that ignores a fraction of extreme training points, then flags test points far from that bulk in Mahalanobis distance—effective when “normal” windows are roughly elliptical in feature space. **This cell:** **PyOD `MCD`** with the notebook’s contamination setting.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.mcd.html" target="_blank" rel="noopener noreferrer">PyOD MCD</a> · <a href="https://scikit-learn.org/stable/modules/covariance.html#robust-covariance-estimation" target="_blank" rel="noopener noreferrer">sklearn robust covariance (Mahalanobis)</a>

In [ ]:
title = 'Minimum Covariance Determinant (MCD)'
try:
    from pyod.models.mcd import MCD
    clf = MCD(contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### One-class SVM (OCSVM)

**Algorithm:** One-class SVM finds a kernelized boundary around the training bulk; points outside the margin are anomalies—useful when normal behavior is not axis-aligned or Gaussian. **This cell:** **PyOD `OCSVM`** on window vectors, distinct from the earlier **sklearn** RBF One-Class SVM used as an ABOD substitute (different implementation path through PyOD).

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.ocsvm.html" target="_blank" rel="noopener noreferrer">PyOD OCSVM</a> · <a href="https://scikit-learn.org/stable/modules/svm.html#svm-outlier-detection" target="_blank" rel="noopener noreferrer">sklearn SVM outlier detection</a>

In [ ]:
title = 'One-class SVM (OCSVM)'
try:
    from pyod.models.ocsvm import OCSVM
    clf = OCSVM(contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### Principal Component Analysis (PCA)

**Algorithm:** PCA-based outlier detection uses deviation from the main variance subspace—either reconstruction error or feature-bag norms—so windows that do not align with dominant co-movement of temperatures stand out. **This cell:** **PyOD’s PCA detector** with fixed contamination, complementing the separate linear PCA reconstruction substitute used for the VAE slot.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.pca.html" target="_blank" rel="noopener noreferrer">PyOD PCA detector</a>

In [ ]:
title = 'Principal Component Analysis (PCA)'
try:
    from pyod.models.pca import PCA
    clf = PCA(contamination=outliers_fraction, random_state=random_state)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### sklearn substitute for Stochastic Outlier Selection (SOS)

**Original idea (PyOD):** SOS builds a stochastic neighborhood model and propagates outlierness through that graph. **This cell:** we use **SGDOneClassSVM**, a linear one-class boundary trained with a stochastic optimizer. The “stochastic” flavor is in the optimization path, not SOS’s graph; it gives a simple linear separator between typical and atypical windows when a margin is a reasonable approximation.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.sos.html" target="_blank" rel="noopener noreferrer">PyOD SOS</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDOneClassSVM.html" target="_blank" rel="noopener noreferrer">sklearn SGDOneClassSVM</a>

In [ ]:
title = 'sklearn substitute for Stochastic Outlier Selection (SOS) (SGDOneClassSVM)'
from sklearn.linear_model import SGDOneClassSVM

clf = SklearnNegLabelsToPyOD(SGDOneClassSVM(nu=outliers_fraction, random_state=random_state))
run_detector(title, clf)


### sklearn substitute for Locally Selective Combination (LSCP)

**Original idea (PyOD):** LSCP combines several base detectors and focuses on regions where local models disagree or are most selective. **This cell:** we train several **novelty LocalOutlierFactor** models with **different neighbor counts** on the same training windows and **majority-vote** their outlier labels on the test set—combination of local density estimates without importing PyOD’s LSCP (which pulls Numba through shared utilities).

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.lscp.html" target="_blank" rel="noopener noreferrer">PyOD LSCP</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.LocalOutlierFactor.html" target="_blank" rel="noopener noreferrer">sklearn LocalOutlierFactor</a>

In [ ]:
title = 'sklearn substitute for Locally Selective Combination (LSCP) (majority novelty LOF)'
clf = LOFMajorityVoteOutlier(lscp_neighbor_values, outliers_fraction)
run_detector(title, clf)


### Connectivity-Based Outlier Factor (COF)

**Algorithm:** COF measures how “connected” a point is along paths through its neighbors, emphasizing spatial chaining of densities rather than raw *k*-NN distance alone—helpful when outliers sit in structured low-density filaments. **This cell:** **PyOD `COF`** with **`COF_N`** neighbors aligned to the same cap as LOF so training geometry stays valid.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.cof.html" target="_blank" rel="noopener noreferrer">PyOD COF</a>

In [ ]:
title = 'Connectivity-Based Outlier Factor (COF)'
try:
    from pyod.models.cof import COF
    clf = COF(n_neighbors=COF_N, contamination=outliers_fraction)
    run_detector(title, clf)
except Exception as err:
    print(f'{title}: skipped  -  {type(err).__name__}: {err}')

### sklearn substitute for Subspace Outlier Detection (SOD)

**Original idea (PyOD):** SOD searches multiple lower-dimensional projections where a point appears extreme. **This cell:** we apply **TruncatedSVD** to map windows into a small linear subspace (capturing dominant shared variation), then run **novelty LOF** in that subspace—one explicit subspace plus local density, as a lightweight proxy for “find weirdness after dimensionality reduction.”

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.sod.html" target="_blank" rel="noopener noreferrer">PyOD SOD</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html" target="_blank" rel="noopener noreferrer">sklearn TruncatedSVD</a>

In [ ]:
title = 'sklearn substitute for Subspace Outlier Detection (SOD) (TruncatedSVD + novelty LOF)'
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline

n_comp = max(2, min(8, df_data_train.shape[1] - 1))
nn_sod = max(2, min(LOF_N, df_data_train.shape[0] - 1))
pipe = Pipeline(
    [
        ('svd', TruncatedSVD(n_components=n_comp, random_state=random_state)),
        (
            'lof',
            LocalOutlierFactor(n_neighbors=nn_sod, novelty=True, contamination=outliers_fraction),
        ),
    ]
)
clf = SklearnNegLabelsToPyOD(pipe)
run_detector(title, clf)


### sklearn substitute for Variational Auto Encoder

**Original idea (PyOD):** A VAE learns a latent distribution and reconstructs inputs, so large reconstruction error or low ELBO-like scores flag anomalies. **This cell:** we use **PCA reconstruction mean squared error**: a **linear** reconstruction model with a contamination-based threshold on error. It keeps the “reconstruction mismatch” intuition without PyTorch, at the cost of nonlinear latent structure.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.vae.html" target="_blank" rel="noopener noreferrer">PyOD VAE module</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html" target="_blank" rel="noopener noreferrer">sklearn PCA</a> · <a href="https://en.wikipedia.org/wiki/Variational_autoencoder" target="_blank" rel="noopener noreferrer">Variational autoencoder (overview)</a>

In [ ]:
title = 'sklearn substitute for Variational Auto Encoder (PCA reconstruction error)'
n_pca_v = max(2, min(12, window_size - 1))
clf = PCAMSEReconstructionOutlier(contamination, n_components=n_pca_v, random_state=random_state)
run_detector(title, clf)


### sklearn substitute for Auto Encoder

**Original idea (PyOD):** A deep autoencoder compresses and expands data through hidden layers to learn nonlinear manifolds of “normal” windows. **This cell:** we train a **small MLPRegressor** to predict each window from itself (identity target), then threshold **per-window reconstruction MSE** at the desired contamination—nonlinear like a tiny autoencoder, but entirely within scikit-learn and WASM-friendly.

**Learn more (new tab):** <a href="https://pyod.readthedocs.io/en/latest/pyod.models.html#deep-learning-models" target="_blank" rel="noopener noreferrer">PyOD deep / AE-style detectors</a> · <a href="https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html" target="_blank" rel="noopener noreferrer">sklearn MLPRegressor</a>

In [ ]:
title = 'sklearn substitute for Auto Encoder (MLP identity reconstruction)'
clf = MLPIdentityReconstructionOutlier(contamination, random_state=random_state)
run_detector(title, clf)


## Next steps

Tune `window_size` and `contamination`, try your own series, or export scores with `clf.decision_function(df_data_test)` once a model fits.

**More reading (new tab):** <a href="https://pyod.readthedocs.io/en/latest/api_cc.html" target="_blank" rel="noopener noreferrer">PyOD API cheat sheet</a> · <a href="https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_anomaly_comparison.html" target="_blank" rel="noopener noreferrer">sklearn anomaly algorithm comparison (examples)</a>.